In [80]:
import pandas as pd
import numpy as np
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
from sklearn.metrics import f1_score
from collections import defaultdict
from sklearn.model_selection import train_test_split
from pyswarms.single.global_best import GlobalBestPSO
from pyswarms.utils.functions import single_obj as fx
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore")
import nest_asyncio
nest_asyncio.apply()
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
import tensorflow_federated as tff
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall
from imblearn.over_sampling import RandomOverSampler,SMOTE
from sklearn import preprocessing

In [81]:
data = pd.read_csv("bs140513_032310.csv")
#print (data.isnull().sum())

In [82]:
############################################ Simple_Model ######################################
#data = pd.read_csv("bs140513_032310.csv")
#data["gender"].fillna("M",inplace=True)
#data["age"].fillna(5,inplace=True)
data["gender"] = data["gender"].map({"M": 1, "F": 2, "E":3,"U":-1})
data["gender"].fillna(-1,inplace=True)
#data.fillna(-999,inplace=True)
# Create two dataframes with fraud and non-fraud data 
df_fraud = data.loc[data.fraud == 1] 
df_non_fraud = data.loc[data.fraud == 0]
pd.concat([df_fraud.groupby('category')['amount'].mean(),df_non_fraud.groupby('category')['amount'].mean(),\
           data.groupby('category')['fraud'].mean()*100],keys=["Fraudulent","Non-Fraudulent","Percent(%)"],axis=1,\
          sort=False).sort_values(by=['Non-Fraudulent'])

# dropping zipcodeori and zipMerchant since they have only one unique value
data_reduced = data.drop(['zipcodeOri','zipMerchant'],axis=1)

In [83]:
# turning object columns type to categorical for easing the transformation process
col_categorical = data_reduced.select_dtypes(include= ['object']).columns
for col in col_categorical:
    data_reduced[col] = data_reduced[col].astype('category')
 #   categorical values ==> numeric values
data_reduced[col_categorical] = data_reduced[col_categorical].apply(lambda x: x.cat.codes)
le = preprocessing.LabelEncoder()
#data_reduced["merchant"]= le.fit_transform(data_reduced["merchant"].values)
data_reduced["customer"]= le.fit_transform(data_reduced["customer"].values)
alice_df = data_reduced[:len(data.index)//2]
bob_df = data_reduced[len(data.index)//2:]

In [84]:
X = data_reduced.drop(['fraud'],axis=1)
y = data['fraud']
#(np.isnan(X))
print (X.isnull().sum())

step        0
customer    0
age         0
gender      0
merchant    0
category    0
amount      0
dtype: int64


In [85]:
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)
y_res = pd.DataFrame(y_res)

In [86]:
# I won't do cross validation since we have a lot of instances
X_train, X_test, y_train, y_test = train_test_split(X_res,y_res,test_size=0.3,random_state=42,shuffle=True,stratify=y_res)

In [87]:
 X_train.shape[1]

7

In [137]:
def input_spec():
    return (
        tf.TensorSpec([None, X_train.shape[1]], tf.float64),
        tf.TensorSpec([None, 1], tf.int64)
    )
def build_model():
    
    final_model =  tf.keras.models.Sequential([
        tf.keras.layers.InputLayer(input_shape=(X_train.shape[1],)),
        #tf.keras.layers.Dropout(0.2),
        #tf.keras.layers.Dense(32, activation='relu'),
        #tf.keras.layers.Dense(64, activation='relu'),
        #tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(4, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])

    return final_model

def compile_model():
    """
    Compile a given keras model using SparseCategoricalCrossentropy
    loss and the Adam optimizer with set evaluation metrics.
    """
    keras_model=build_model()
    keras_model.compile(
        loss=tf.keras.losses.BinaryCrossentropy(),
        #optimizer=optimizer.optimize(fx,10),
        optimizer=tf.keras.optimizers.Adam(),

        metrics=[BinaryAccuracy()]
    )

    return keras_model

In [138]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

    model = compile_model()

    final_keras_model =model
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [90]:
from sklearn import preprocessing
def make_tf_dataset(input_data,  batch_size=None):
    # Create two dataframes with fraud and non-fraud data 

# dropping zipcodeori and zipMerchant since they have only one unique value
    #data_reduced = input_data.drop(['zipcodeOri','zipMerchant','customer','gender',"merchant"],axis=1)
    X = input_data.drop(['fraud'],axis=1)
    y =input_data['fraud']
    scaler=StandardScaler()
    X= scaler.fit_transform(X)
    sm = SMOTE(random_state=42)
    X_res, y_res = sm.fit_resample(X, y)
    y_res = pd.DataFrame(y_res)
    #X=input_data.drop('isFraud',axis=1)
    # I won't do cross validation since we have a lot of instances
    X_train, X_test, y_train, y_test = train_test_split(X_res,y_res,test_size=0.3,random_state=42)
   # x = (data[["type", "amount", "oldbalanceOrg", "newbalanceOrig"]])
#y = (data[["isFraud"]])
#x = np.asarray(x).astype('float32')
#y = np.asarray(y)
   # X_train =tf.convert_to_tensor(X_train, dtype=tf.float64)
    #y_train = tf.convert_to_tensor(y_train, dtype=tf.int64)
   
    #X_train = np.asarray(X_train).astype(np.float32)
    #y_train = np.asarray(y_train).astype(np.float32)
    #tf.convert_to_tensor(X_train, dtype=tf.float32)
    #scaler = MinMaxScaler()
    #X_train = scaler.fit_transform(X_train)
    #sm = SMOTE(random_state=42)
    #X_train, y_train = sm.fit_resample(X_train, y_train)
    # Dataset creation
    dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    dataset = dataset.shuffle(2048, seed=SEED)
    if batch_size:
        dataset = dataset.batch(batch_size)

    return dataset
EPOCHS = 100
BATCH_SIZE = 64
SEED = 1337
train_data, val_data = [], []

for client_data in [alice_df, bob_df]:
    train_df, val_df = train_test_split(client_data, test_size=0.2, random_state=SEED)

      # TF Datasets
    train_data.append(make_tf_dataset(train_df, batch_size=BATCH_SIZE))
    val_data.append(make_tf_dataset(val_df, batch_size=1))

In [97]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.1),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=1.0)
)

EPOCHS=100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 351.5399684906006


In [98]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.95933753),
             ('precision', 0.9455849),
             ('recall', 0.975076),
             ('loss', 0.12302651)])

In [139]:
####################################### HHO_FL_CCFD ####################################################
def evaluate_model(model, X, y):
    # Evaluate the model on the given data
    y_pred = model.predict(X)
    y_pred = (y_pred > 0.5).astype(int)
    f1 = f1_score(y, y_pred)

    return f1

def objective_function(w):
    # Set the weights of the local model
    shape = get_shape(model)
    model.set_weights((set_shape(w,shape)))

    # Evaluate the local model on the local data
    f1_local = evaluate_model(model, X_train, y_train)

    return -f1_local

def hho_federated_learning(f, n, m, alpha, beta, gamma, lb, ub, iter_max):
    """
    Harris Hawks Optimization for Federated Learning

    Parameters:
    f (function): Objective function to optimize
    n (int): Number of Harris hawks
    m (int): Number of weights in each local model
    alpha (float): Harris hawks' step size
    beta (float): Harris hawks' switching probability
    gamma (float): Harris hawks' shrinking coefficient
    lb (ndarray): Lower bounds of the search space
    ub (ndarray): Upper bounds of the search space
    iter_max (int): Maximum number of iterations

    Returns:
    g_best (float): Global best objective value
    x_best (ndarray): Global best solution
    """
    # Initialize Harris hawks
    x = np.random.uniform(lb, ub, (n, m))
    f_x = np.apply_along_axis(f, 1, x)
    g_best = np.min(f_x)
    x_best = x[np.argmin(f_x)]

    # Main loop
    for t in range(iter_max):
        print(t)
        # Calculate fitness values
        fitness = 1 / (1 + f_x)

        # Sort Harris hawks by fitness values
        idx = np.argsort(fitness)[::-1]
        x = x[idx]
        fitness = fitness[idx]

        # Update Harris hawks' positions
        for i in range(n):
            if np.random.rand() < beta:
                # Exploration: move towards the global best solution
                x[i] = x[i] + alpha * (x_best - x[i]) + gamma * np.random.randn(m)
            else:
                # Exploitation: move towards the best solution in the neighborhood
                j = np.random.choice(np.delete(np.arange(n), i))
                x[i] = x[i] + alpha * (x[j] - x[i]) + gamma * np.random.randn(m)

            # Apply bounds to the positions
            x[i] = np.clip(x[i], lb, ub)

        # Evaluate new positions
        f_x = np.apply_along_axis(f, 1, x)
        g_best_new = np.min(f_x)
        x_best_new = x[np.argmin(f_x)]

        # Update global best solution
        if g_best_new < g_best:
            g_best = g_best_new
            x_best = x_best_new

    return g_best, x_best




# Build a neural network model
model = compile_model()

# Set HHO parameters
n = 10
m = model.count_params()
alpha = 0.2
beta = 0.8
gamma = 0.3
lb = np.full(m, -1)
ub = np.full(m, 1)
def get_shape(model):
    weights_layer = model.get_weights()
    shapes = []
    for weights in weights_layer:
        shapes.append(weights.shape)
    return shapes
def set_shape(weights,shapes):
    new_weights = []
    index=0
    for shape in shapes:
        if(len(shape)>1):
            n_nodes = np.prod(shape)+index
        else:
            n_nodes=shape[0]+index
        tmp = np.array(weights[index:n_nodes]).reshape(shape)
        new_weights.append(tmp)
        index=n_nodes
    return new_weights
import time
start = time.time()
hho_federated_learning(objective_function, n, m, alpha, beta, gamma, lb, ub, 10)
end=time.time()
print(f"Runtime of the program is {end - start}")

0
1
2
3
4
5
6
7
8
9
Runtime of the program is 849.9758024215698


In [100]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

   # keras_model = build_model()

    final_keras_model =model
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [112]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.1),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=1.0)
)

EPOCHS=100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 271.87024688720703


In [113]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.9545952),
             ('precision', 0.9725282),
             ('recall', 0.93595135),
             ('loss', 0.14476834)])

In [140]:
############################### ACO_FL_CCFD ###############################
def get_shape(model):
    weights_layer = model.get_weights()
    shapes = []
    for weights in weights_layer:
        shapes.append(weights.shape)
    return shapes
def set_shape(weights,shapes):
    new_weights = []
    index=0
    for shape in shapes:
        if(len(shape)>1):
            n_nodes = np.prod(shape)+index
        else:
            n_nodes=shape[0]+index
        tmp = np.array(weights[index:n_nodes]).reshape(shape)
        new_weights.append(tmp)
        index=n_nodes
    return new_weights
def evaluate_model(model, X, y):
    # Evaluate the model on the given data
    y_pred = model.predict(X)
    y_pred = (y_pred > 0.5).astype(int)
    f1 = f1_score(y, y_pred)

    return f1

def objective_function(w,model):
    # Set the weights of the local model
    shape = get_shape(model)
    model.set_weights((set_shape(w,shape)))

    # Evaluate the local model on the local data
    f1_local = evaluate_model(model, X_train, y_train)

    return -f1_local


def normalize(probabilities):
    total = sum(probabilities)
    if total == 0:
        return [1 / len(probabilities)] * len(probabilities)
    return [p / total for p in probabilities]

def aco_fl(objective_function, num_ants, num_iterations, q0, alpha, beta, rho, lb, ub, global_model,min_pheromone=1e-12, max_pheromone=1e6, scale_pheromone=1):
    # Initialize the pheromone matrix and the probability matrix
    pheromone = np.ones_like(lb) * scale_pheromone
    probability = np.ones_like(lb) / len(lb)
    best_position = np.zeros_like(lb)
    best_fitness = objective_function(best_position,  global_model)
    #pheromone = np.ones_like(lb)
    #probability = np.ones_like(lb)/len(lb)
    #best_position = np.zeros_like(lb)
    #best_fitness = objective_function(best_position,  global_model)
    
    # Main loop of the ACO algorithm
    for i in range(num_iterations):
        # Initialize the ant solutions
        print('“---- Iteration {} ----”'.format(i + 1))
        print('“Pheromone:”', pheromone)
        print('“Probability:”', probability)
        print('“Current position:”', best_position)
        solutions = np.zeros((num_ants, len(lb)))
        fitness = np.zeros(num_ants)
        
        # Construct the ant solutions
        for j in range(num_ants):
            current_position = np.zeros_like(lb)
            for k in range(len(lb)):
                # Choose the next hyperparameter using the probability matrix and the pheromone matrix
                if np.random.rand() < q0:
                    next_position = np.argmax(probability)
                else:
                    
                    probabilities = pheromone**alpha * probability**beta
                    probabilities = probabilities/np.sum(probabilities)
                    probabilities = normalize(probabilities)
                    if np.isnan(probabilities).any():
                        probabilities = [1 / len(probabilities)] * len(probabilities)
                    #probabilities = np.divide(probabilities, np.sum(probabilities), out=np.zeros_like(probabilities), where=np.sum(probabilities) != 0)
                    #probabilities =np.nan_to_num(probabilities ,nan=0)
                    #probabilities = np.nan_to_num(probabilities)
                    #print('“Before normalization:”', probabilities)  # Add this print statement
                    
                    #print('“After normalization:”', probabilities)  # Add this print statement
                   # probabilities = normalize(probabilities)
                    next_position = np.random.choice(len(lb), p=probabilities)
                current_position[next_position] = 1
                
            # Evaluate the fitness of the ant solution
            solutions[j] = lb + (np.array(ub)-np.array(lb))*current_position
            fitness[j] = objective_function(solutions[j],  global_model)
            
            # Update the global best solution
            if fitness[j] < best_fitness:
                best_fitness = fitness[j]
                best_position = solutions[j]
        
        # Update the pheromone matrix


        delta_pheromone = np.zeros_like(pheromone)
        for j in range(num_ants):
            for k in range(len(lb)):
                if solutions[j][k] == 1:
                    epsilon = 1e-9
        
                    #delta_pheromone=pd.DataFrame(delta_pheromone)
                    #fitness = fitness.replace([np.inf, -np.inf, -0], 0)
                    delta_pheromone[k] += 1/(fitness[j]+epsilon)

        pheromone = (1-rho)*pheromone + rho*delta_pheromone           
        

        pheromone = np.clip(pheromone, min_pheromone, max_pheromone)

        
        # Update the probability matrix
        probability = pheromone**alpha * probability**beta
        probability = probability/np.sum(probability)


    return best_position, best_fitness


# Define the global model
#global_model = build_model()
#global_model.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['accuracy'])



q0=0.2
alpha=0.7
beta=0.1
rho=0.2
# Run the ACO algorithm
num_ants = 3
num_iterations = 5
global_model=compile_model()
m = global_model.count_params()
lb = np.full(m, -1)
ub = np.full(m, 1)
import time
start = time.time()
best_position, best_fitness = aco_fl(objective_function, num_ants, num_iterations, q0, alpha, beta, rho, lb, ub,global_model,min_pheromone=1e-12, max_pheromone=1e6, scale_pheromone=1)
end=time.time()
print(f"Runtime of the program is {end - start}")

“---- Iteration 1 ----”
“Pheromone:” [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
“Probability:” [0.02702703 0.02702703 0.02702703 0.02702703 0.02702703 0.02702703
 0.02702703 0.02702703 0.02702703 0.02702703 0.02702703 0.02702703
 0.02702703 0.02702703 0.02702703 0.02702703 0.02702703 0.02702703
 0.02702703 0.02702703 0.02702703 0.02702703 0.02702703 0.02702703
 0.02702703 0.02702703 0.02702703 0.02702703 0.02702703 0.02702703
 0.02702703 0.02702703 0.02702703 0.02702703 0.02702703 0.02702703
 0.02702703]
“Current position:” [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
“---- Iteration 2 ----”
“Pheromone:” [1.e-12 1.e-12 1.e-12 6.e-01 1.e-12 8.e-01 1.e-12 8.e-01 4.e-01 1.e-12
 1.e-12 1.e-12 1.e-12 8.e-01 1.e-12 4.e-01 1.e-12 1.e-12 1.e-12 1.e-12
 8.e-01 1.e-12 4.e-01 4.e-01 6.e-01 1.e-12 4.e-01 1.e-12 1.e-12 1.e-12
 1.e-12 4.e-01 4.e-01 1.e-12 4.e-01 4.e-01 4.e-01]
“Probability:” [3.94719843e-10 3.94719843e-10 3.94719843e-10

In [115]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

   # keras_model = build_model()

    final_keras_model =global_model
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [118]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.1),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=1.0)
)

EPOCHS=100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 274.05650186538696


In [119]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.9591734),
             ('precision', 0.9476512),
             ('recall', 0.97234976),
             ('loss', 0.12007755)])

In [120]:
def get_shape(model):
    weights_layer = model.get_weights()
    shapes = []
    for weights in weights_layer:
        shapes.append(weights.shape)
    return shapes
def set_shape(weights,shapes):
    new_weights = []
    index=0
    for shape in shapes:
        if(len(shape)>1):
            n_nodes = np.prod(shape)+index
        else:
            n_nodes=shape[0]+index
        tmp = np.array(weights[index:n_nodes]).reshape(shape)
        new_weights.append(tmp)
        index=n_nodes
    return new_weights

In [121]:
def evaluate_model(model, X, y):
    # Evaluate the model on the given data
    y_pred = model.predict(X)
    y_pred = (y_pred > 0.5).astype(int)
    f1 = f1_score(y, y_pred)

    return f1

def objective_function(w,model):
    # Set the weights of the local model
    shape = get_shape(model)
    model.set_weights((set_shape(w,shape)))

    # Evaluate the local model on the local data

    f1_local = evaluate_model(model, X_train, Y_train)

    return -f1_local

In [141]:
def evaluate_nn(W, shape,X_train=X_train, Y_train=y_train):
    results = []

    for weights in W:
        model.set_weights(set_shape(weights,shape))
        score = model.evaluate(X_train, Y_train, verbose=0)
        results.append(1-score[1])
    return results
model=compile_model()
shape = get_shape(model)
model=compile_model()
m = model.count_params()
x_max = 1.0 * np.ones(m)
x_min = -1.0 * x_max
bounds = (x_min, x_max)
options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
optimizer = GlobalBestPSO(n_particles=5, dimensions=m,
                          options=options, bounds=bounds)
import time
start = time.time()
cost, pos = optimizer.optimize(evaluate_nn, 10, X_train=X_train, Y_train=y_train,shape=shape)
model.set_weights(set_shape(pos,shape))
end=time.time()
print(f"Runtime of the program is {end - start}")

2023-06-13 17:09:23,779 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|10/10, best_cost=0.175
2023-06-13 17:17:49,066 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.17510640621185303, best pos: [-0.95139304 -0.89864729 -0.80285083 -0.18034003 -0.34310408  0.23537922
  0.25775766 -0.04410789 -0.22372766 -0.95918695  0.67498901 -0.6200617
  0.50623231 -0.08468947 -0.44802871  0.92360921  0.77291981  0.58306926
  0.8240975  -0.20905773  0.54136969 -0.24946917 -0.53477677  0.02726907
 -0.753767    0.84316733 -0.79610204  0.07689006  0.9979636   0.44833531
  0.41767415 -0.36726598  0.60568307  0.63428336 -0.86462095  0.42937528
  0.74737442]


Runtime of the program is 505.29278206825256


In [123]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

   # keras_model = build_model()

    final_keras_model =model
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [124]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.1),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=1.0)
)

EPOCHS=100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 294.83799481391907


In [125]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.9424962),
             ('precision', 0.9548244),
             ('recall', 0.929372),
             ('loss', 0.13808481)])

In [142]:
############################### CSO_FL_CCFD ###############################
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
from sklearn.metrics import f1_score
from collections import defaultdict
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE


# Define the objective function
def evaluate_model(model, X, y):
    # Evaluate the model on the given data
    y_pred = model.predict(X)
    y_pred = (y_pred > 0.5).astype(int)
    f1 = f1_score(y, y_pred)

    return f1

def objective_function(w,model):
    # Set the weights of the local model
    shape = get_shape(model)
    model.set_weights((set_shape(w,shape)))

    # Evaluate the local model on the local data
    f1_local = evaluate_model(model, X_train, y_train)

    return -f1_local

# Define the CSO algorithm with federated learning
def cso_fl(objective_function, num_cats, num_iterations, lb, ub,  global_model):
    # Initialize the population of cats with random positions and velocities
    positions = np.random.uniform(lb, ub, size=(num_cats, len(lb)))
    velocities = np.zeros_like(positions)
    best_position = positions[0]
    best_fitness = objective_function(positions[0],  global_model)
    
    # Main loop of the CSO algorithm
    for i in range(num_iterations):
        # Evaluate the fitness of each cat
        fitness = np.array([objective_function(p,  global_model) for p in positions])
        
        # Update the global best position
        index = np.argmin(fitness)
        if fitness[index] < best_fitness:
            best_fitness = fitness[index]
            best_position = positions[index]
        
        # Update the position and velocity of each cat
        for j in range(num_cats):
            # Update the velocity of the cat
            r1 = np.random.random(size=velocities.shape[1])
            r2 = np.random.random(size=velocities.shape[1])
            velocities[j] = velocities[j] + r1*(best_position - positions[j]) + r2*(positions[index] - positions[j])
            
            # Update the position of the cat
            positions[j] = positions[j] + velocities[j]
            
            # Check the boundaries of the position
            positions[j] = np.clip(positions[j], lb, ub)
    
    return best_position, best_fitness



#client_data = [(x_train[i::num_clients], y_train[i::num_clients]) for i in range(num_clients)]
# Define the global model
global_model = compile_model()
#global_model.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['accuracy'])

# Set the hyperparameter search space

m = global_model.count_params()
lb = np.full(m, -1)
ub = np.full(m, 1)

# Run the CSO algorithm
num_cats = 3
num_iterations = 5
import time
start = time.time()
best_position, best_fitness = cso_fl(objective_function, num_cats, num_iterations, lb, ub,  global_model)
end=time.time()
print(f"Runtime of the program is {end - start}")

Runtime of the program is 124.58059644699097


In [127]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

   # keras_model = build_model()

    final_keras_model =global_model
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [128]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.1),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=1.0)
)

EPOCHS=100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 286.5268220901489


In [129]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.9578905),
             ('precision', 0.9593312),
             ('recall', 0.95663446),
             ('loss', 0.12714785)])

In [130]:
def input_spec():
    return (
        tf.TensorSpec([None, X_train.shape[1]], tf.float64),
        tf.TensorSpec([None, 1], tf.int64)
    )
def build_model(learning_rate, num_filters):
    
    final_model =  tf.keras.models.Sequential([
        tf.keras.layers.InputLayer(input_shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(num_filters, activation='relu'),
        tf.keras.layers.Dense(num_filters*2, activation='relu'),
        #tf.keras.layers.Dense(num_filters, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    
    
    final_model.compile(
        loss=tf.keras.losses.BinaryCrossentropy(),
        #optimizer=optimizer.optimize(fx,10),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=[BinaryAccuracy(), Precision(), Recall()]
    )


    """
    Compile a given keras model using SparseCategoricalCrossentropy
    loss and the Adam optimizer with set evaluation metrics.
    """

   
    return final_model


In [135]:
import heapq
class HeapBasedOptimizer:
    def __init__(self):
        self.performance_heap = []

    def add_model(self, model, performance):
        heapq.heappush(self.performance_heap, (-performance, model))

    def get_best_model(self):
        return heapq.heappop(self.performance_heap)[1]
import time
start = time.time()
# Heap based optimizer
optimizer = HeapBasedOptimizer()

# Define range for hyperparameters
learning_rates = [0.001, 0.0005]
num_filters_values = [32]

# Optimize the CNN model
epochs = 10
for learning_rate in learning_rates:
    for num_filters in num_filters_values:
        model = build_model(learning_rate, num_filters)
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0)
        performance = max(history.history["val_binary_accuracy"])  # You can use other performance metrics if desired
        optimizer.add_model(model, performance)

# Get the best model from the heap
best_model = optimizer.get_best_model()
end=time.time()
print(f"Runtime of the program is {end - start}")

Runtime of the program is 461.1085388660431


In [132]:
def model_fn():
    """
    Create TFF model from compiled Keras model and a sample batch.
    """

   # keras_model = build_model()

    final_keras_model =best_model 
    keras_model_clone = tf.keras.models.clone_model(final_keras_model)

    return tff.learning.from_keras_model(keras_model_clone,     input_spec=input_spec(),
        loss=tf.keras.losses.BinaryCrossentropy(),metrics=[BinaryAccuracy(), Precision(), Recall()])

In [133]:
trainer = tff.learning.build_federated_averaging_process(
    model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=0.1),
    server_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=1.0)
)

EPOCHS=100
import time
start = time.time()
state = trainer.initialize()
#state.model.assign_weights_to(model_fn)
train_hist = []
for i in range(EPOCHS):
    state, metrics = trainer.next(state, train_data)
    train_hist.append(metrics)

    print(f"\rRun {i+1}/{EPOCHS}", end="")
end=time.time()
print(f"Runtime of the program is {end - start}")

Run 100/100Runtime of the program is 370.8435456752777


In [134]:
evaluator = tff.learning.build_federated_evaluation(model_fn)
federated_metrics = evaluator(state.model, val_data)
federated_metrics

OrderedDict([('binary_accuracy', 0.9626511),
             ('precision', 0.9780704),
             ('recall', 0.94679576),
             ('loss', 0.13443159)])